# 06 — European Greeks & Hessians

Compute option sensitivities via **JAX automatic differentiation**:
- First-order Greeks: Delta, Vega, Theta, Rho
- Second-order Greeks: Gamma, Vanna, Volga, Charm
- Full **4×4 Hessian matrix** in a single `jax.hessian` call
- Comparison to analytic BSM formulae and QuantLib's FD bumping

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms, plot_heatmap
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.engines.analytic.black_formula import black_scholes_price

S, K, T = 100.0, 100.0, 1.0
r, q, sigma = 0.05, 0.02, 0.20
CALL = 1

---
## 1. First-Order Greeks

| Greek | Definition | AD |
|---|---|---|
| Delta | $\frac{\partial V}{\partial S}$ | `jax.grad(f, 0)` |
| Vega | $\frac{\partial V}{\partial \sigma}$ | `jax.grad(f, 1)` |
| Rho | $\frac{\partial V}{\partial r}$ | `jax.grad(f, 2)` |
| Theta | $-\frac{\partial V}{\partial T}$ | `-jax.grad(f, 3)` |

In [ ]:
def bsm(spot, vol, rate, ttm):
    return black_scholes_price(spot, K, ttm, rate, q, vol, CALL)

args = tuple(jnp.float64(x) for x in [S, sigma, r, T])

# All first-order Greeks in one call
grad_fn = jax.grad(bsm, argnums=(0, 1, 2, 3))
delta_ad, vega_ad, rho_ad, dVdT_ad = [float(g) for g in grad_fn(*args)]
theta_ad = -dVdT_ad  # theta = -dV/dT

print("First-Order Greeks (AD):")
print(f"  Delta : {delta_ad:+.8f}")
print(f"  Vega  : {vega_ad:+.8f}")
print(f"  Rho   : {rho_ad:+.8f}")
print(f"  Theta : {theta_ad:+.8f}")

In [ ]:
# ── QuantLib FD Greeks ──────────────────────────────────────────────────────
ql_today = QL.Date(15, 1, 2024)
QL.Settings.instance().evaluationDate = ql_today

ql_spot   = QL.QuoteHandle(QL.SimpleQuote(S))
ql_r_ts   = QL.YieldTermStructureHandle(QL.FlatForward(ql_today, r, QL.Actual365Fixed()))
ql_q_ts   = QL.YieldTermStructureHandle(QL.FlatForward(ql_today, q, QL.Actual365Fixed()))
ql_vol_ts = QL.BlackVolTermStructureHandle(QL.BlackConstantVol(ql_today, QL.NullCalendar(), sigma, QL.Actual365Fixed()))
ql_process = QL.BlackScholesMertonProcess(ql_spot, ql_q_ts, ql_r_ts, ql_vol_ts)

exercise = QL.EuropeanExercise(ql_today + QL.Period(1, QL.Years))
payoff   = QL.PlainVanillaPayoff(QL.Option.Call, K)
ql_opt   = QL.VanillaOption(payoff, exercise)
ql_opt.setPricingEngine(QL.AnalyticEuropeanEngine(ql_process))

ql_delta = ql_opt.delta()
ql_gamma = ql_opt.gamma()
ql_vega  = ql_opt.vega() / 100   # QL vega is per 1% vol shift
ql_theta = ql_opt.theta()
ql_rho   = ql_opt.rho() / 100    # QL rho is per 1% rate shift

compare_table([
    ("Delta", ql_delta, delta_ad),
    ("Vega",  ql_vega,  vega_ad),
    ("Rho",   ql_rho,   rho_ad),
    ("Theta", ql_theta / 365, theta_ad),  # QL theta is per calendar day
], title="First-Order Greeks: QL vs JAX AD")

---
## 2. Second-Order Greeks via Hessian

The **Hessian** $H_{ij} = \frac{\partial^2 V}{\partial x_i \partial x_j}$ gives all second-order sensitivities in one matrix:

$$
H = \begin{pmatrix}
\text{Gamma} & \text{Vanna} & \cdot & \text{Charm} \\
\text{Vanna} & \text{Volga} & \cdot & \cdot \\
\cdot & \cdot & \cdot & \cdot \\
\text{Charm} & \cdot & \cdot & \cdot
\end{pmatrix}
$$

where inputs are $(S, \sigma, r, T)$.

In [ ]:
# Full 4×4 Hessian in one call
hessian_fn = jax.hessian(bsm, argnums=(0, 1, 2, 3))
H_raw = hessian_fn(*args)

# Flatten nested tuple structure into matrix
n = 4
H = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        H[i, j] = float(H_raw[i][j])

param_names = ["S (Spot)", "σ (Vol)", "r (Rate)", "T (Time)"]
greek_names = {
    (0,0): "Gamma", (0,1): "Vanna", (1,0): "Vanna",
    (1,1): "Volga", (0,3): "Charm", (3,0): "Charm",
}

print("4×4 Hessian Matrix:")
print(f"{'':>10s}", end="")
for name in param_names:
    print(f"{name:>14s}", end="")
print()
for i, name in enumerate(param_names):
    print(f"{name:>10s}", end="")
    for j in range(n):
        print(f"{H[i,j]:>14.6f}", end="")
    print()

print(f"\nKey second-order Greeks:")
print(f"  Gamma (∂²V/∂S²)   : {H[0,0]:.8f}")
print(f"  Vanna (∂²V/∂S∂σ)  : {H[0,1]:.8f}")
print(f"  Volga (∂²V/∂σ²)   : {H[1,1]:.8f}")
print(f"  Charm (∂²V/∂S∂T)  : {H[0,3]:.8f}")

In [ ]:
# Verify Gamma against QuantLib
compare_table([
    ("Gamma (∂²V/∂S²)", ql_gamma, H[0, 0]),
], title="Second-Order: QL Gamma vs JAX Hessian")

In [ ]:
# Hessian heatmap
plot_heatmap(
    H,
    xlabels=["S", "σ", "r", "T"],
    ylabels=["S", "σ", "r", "T"],
    title="BSM Option Hessian: ∂²V/∂xᵢ∂xⱼ",
    fmt=".4f"
)

---
## 3. Greek Surfaces

Plot Delta and Gamma as functions of spot and time-to-maturity.

In [ ]:
import matplotlib.pyplot as plt

spots = jnp.linspace(70, 130, 60)
ttms  = jnp.linspace(0.01, 2.0, 60)

# Vectorized delta & gamma over spots for fixed T
def delta_fn(s):
    return jax.grad(lambda s_: black_scholes_price(s_, K, T, r, q, sigma, CALL))(s)

def gamma_fn(s):
    return jax.grad(jax.grad(lambda s_: black_scholes_price(s_, K, T, r, q, sigma, CALL)))(s)

deltas = jax.vmap(delta_fn)(spots)
gammas = jax.vmap(gamma_fn)(spots)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.array(spots), np.array(deltas), 'b-', linewidth=2)
ax1.axvline(K, color='k', linestyle=':', alpha=0.5)
ax1.set_xlabel('Spot')
ax1.set_ylabel('Delta')
ax1.set_title('Delta vs Spot (via jax.grad + vmap)')
ax1.grid(True, alpha=0.3)

ax2.plot(np.array(spots), np.array(gammas), 'r-', linewidth=2)
ax2.axvline(K, color='k', linestyle=':', alpha=0.5)
ax2.set_xlabel('Spot')
ax2.set_ylabel('Gamma')
ax2.set_title('Gamma vs Spot (via jax.hessian + vmap)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 4. Performance: AD vs FD Bumping

In [ ]:
# FD bumping (4 inputs → 4 bumps)
def fd_greeks():
    base = float(black_scholes_price(S, K, T, r, q, sigma, CALL))
    eps = 1e-4
    d = (float(black_scholes_price(S + eps, K, T, r, q, sigma, CALL)) - base) / eps
    v = (float(black_scholes_price(S, K, T, r, q, sigma + eps, CALL)) - base) / eps
    rh = (float(black_scholes_price(S, K, T, r + eps, q, sigma, CALL)) - base) / eps
    th = -(float(black_scholes_price(S, K, T - eps, r, q, sigma, CALL)) - base) / eps
    return d, v, rh, th

# AD (all Greeks in one reverse pass)
jit_grad = jax.jit(jax.grad(bsm, argnums=(0, 1, 2, 3)))
_ = jit_grad(*args)  # warmup

fd_ms, _ = timed_ms(fd_greeks, warmup=3, runs=20)
ad_ms, _ = timed_ms(lambda: jit_grad(*args), warmup=3, runs=20)

print(f"4 Greeks:")
print(f"  FD bumping (4 reprices) : {fd_ms:.4f} ms")
print(f"  AD reverse mode (1 pass): {ad_ms:.4f} ms")
print(f"  Speedup                 : {fd_ms/ad_ms:.1f}×")

---
## 5. Exercises

1. Compute the **6×6 Hessian** for a Heston-priced option with inputs $(S, K, v_0, \kappa, \theta, \rho)$.
2. Verify **Hessian symmetry**: $H_{ij} = H_{ji}$ for all pairs.
3. Plot a **3D surface** of Vanna (∂²V/∂S∂σ) as a function of spot and vol.

---
*End of Notebook 06*